# Week 7 — Irregular Time Series, Periodicity, Red Noise and Gaussian Processes

**Contact:** 3 hours lecture  
**Course:** MSc Astrostatistics for the Survey Era

## Learning notebooks from the Crete school
- `../../08_Gaussian_Processes/Gaussian_Processes.ipynb`

These Crete notebooks are **learning material**. This lecture notebook supplies the semester narrative, deeper astronomical interpretation, and the binary-SMBH case study.

## Three-hour pacing
- **0:00–0:20** — recap, motivating scientific question, diagnostic poll
- **0:20–1:05** — theory block I + worked derivation
- **1:05–1:15** — short break / discussion
- **1:15–2:00** — theory block II + computational demonstration
- **2:00–2:10** — short break
- **2:10–2:45** — binary-SMBH case study / failure mode
- **2:45–3:00** — synthesis, exit questions, bridge to the 4-hour practical

## Learning objectives

- Explain why irregular sampling changes time-series inference.
- Interpret a sampling window and distinguish aliases from physical periods.
- Compare white-noise, periodic and stochastic red-noise models.
- Write the Gaussian-process likelihood and interpret kernels physically.
- Use simulated truth to validate DRW/DHO and periodicity recovery.

## Unit convention for this dataset

Three time quantities appear in different units:

$$
P_{\rm orb,obs}\quad [\mathrm{yr}],
\qquad
t_{\rm LC}\quad [\mathrm{MJD\ days}],
\qquad
\tau_{\rm DRW/DHO}\quad [\mathrm{days}].
$$

Therefore, when a period inferred from a light curve is compared with the
stored physical truth, make the unit conversion explicit. For example,

$$
P_{\rm days}=365\,P_{\rm yr}
$$

when following the same 365-day conversion used in the provider notebook.

The stochastic-process truth fields are unavailable in the provider's blinded
dataset and should remain hidden until validation.


## 1. Astronomical time series are not regular textbook signals


Survey light curves are irregular, seasonal, heteroscedastic and often sparse. If the observation times are $t_i$, the measured process is effectively multiplied by a sampling window

$$
W(t)=\sum_i \delta(t-t_i).
$$

Structure in the window can create aliases and complicated frequency responses. A periodogram peak therefore combines information about the underlying process and the sampling pattern.

The binary-SMBH simulations preserve realistic per-band time arrays, which makes them especially valuable for teaching this point. Students can analyse the same physical system under different band cadences and see how inference changes.

> **Discussion checkpoint.** Identify the estimand, observed data, and strongest modelling assumption in the example just discussed.

### Worked computational demonstration — Simulate irregular stochastic + periodic data

Run the cell, then change one assumption and explain what changes statistically.

In [ ]:
import numpy as np, matplotlib.pyplot as plt
rng=np.random.default_rng(2026)
t=np.sort(rng.uniform(0,500,180))
# correlated OU-like process on the irregular grid
tau,sig=80.,0.12
K=sig**2*np.exp(-np.abs(t[:,None]-t[None,:])/tau)
red=rng.multivariate_normal(np.zeros(len(t)),K)
period=120.
y=20 + red + .08*np.sin(2*np.pi*t/period)
err=rng.uniform(.015,.05,len(t))
yobs=y+rng.normal(0,err)
plt.errorbar(t,yobs,yerr=err,fmt='.',ms=3)
plt.gca().invert_yaxis(); plt.xlabel('time'); plt.ylabel('mag'); plt.show()

## 2. Periodic models and Lomb-Scargle


For unevenly sampled data, Lomb-Scargle can be viewed as repeated fitting of sinusoidal models over frequency. A simple model is

$$
m(t)=C+A\cos\omega t+B\sin\omega t.
$$

The highest periodogram peak is not automatically the period. One must consider frequency range, the number of effectively searched frequencies, the window function and the stochastic null process.

When the source has red noise, false-alarm probabilities derived under independent white noise can be severely over-optimistic.

### Worked computational demonstration — Lomb-Scargle-style scan with scipy

Run the cell, then change one assumption and explain what changes statistically.

In [ ]:
from scipy.signal import lombscargle
freq=np.linspace(1/400,1/10,5000)
ang=2*np.pi*freq
yy=yobs-np.mean(yobs)
power=lombscargle(t,yy,ang,normalize=True)
best=1/freq[np.argmax(power)]
print('highest periodogram period:',best)
plt.plot(1/freq,power); plt.xscale('log'); plt.xlabel('period'); plt.ylabel('power'); plt.show()

## 3. Red noise, DRW and DHO


AGN variability is often represented by stochastic processes. A damped random walk / Ornstein-Uhlenbeck covariance can be written schematically as

$$
k(\Delta t)=\sigma^2\exp(-|\Delta t|/\tau),
$$

with amplitude $\sigma$ and characteristic time $\tau$. Damped harmonic oscillator models can represent more complex power spectral shapes and quasi-oscillatory behaviour.

The course dataset stores true DRW/DHO parameters per band. This creates an important validation exercise: fit the observable light curves without using these columns, then compare recovered quantities with the hidden truth.

### Worked computational demonstration — GP log likelihood helper

Run the cell, then change one assumption and explain what changes statistically.

In [ ]:
def gp_loglike(t,y,err,amp,tau,mean=None):
    mean=np.mean(y) if mean is None else mean
    K=amp**2*np.exp(-np.abs(t[:,None]-t[None,:])/tau)
    K[np.diag_indices_from(K)] += err**2
    L=np.linalg.cholesky(K)
    r=y-mean
    alpha=np.linalg.solve(L.T,np.linalg.solve(L,r))
    return -0.5*r@alpha - np.sum(np.log(np.diag(L))) - 0.5*len(t)*np.log(2*np.pi)

print(gp_loglike(t,yobs,err,.1,80.))

## 4. Gaussian processes as distributions over functions


A Gaussian process specifies that any finite vector of function values follows a multivariate Normal distribution:

$$
\mathbf f\sim N(\mathbf m,K),\qquad K_{ij}=k(t_i,t_j).
$$

With independent Gaussian measurement errors,

$$
K_y=K+\mathrm{diag}(\sigma_i^2).
$$

The GP log marginal likelihood is

$$
\log p(\mathbf y\mid\phi)
=
-\frac12\mathbf r^T K_y^{-1}\mathbf r
-\frac12\log|K_y|
-\frac n2\log(2\pi),
$$

where $\phi$ are kernel hyperparameters and $\mathbf r=\mathbf y-\mathbf m$.

The kernel is a modelling assumption. A flexible kernel can interpolate beautifully while representing the wrong physical process.

## 5. Testing periodicity against a stochastic null


For AGN, the scientifically relevant comparison is often not "sinusoid versus white noise" but "stochastic red-noise model versus stochastic+periodic model." Posterior predictive simulations from the red-noise null allow the analyst to ask how often similar apparent periodic structure arises without a true periodic component.

This week should leave students suspicious of isolated periodogram peaks and comfortable with the idea that detection significance depends on the null model.

## 6. Multiband information


The $u,g,r,i,z,y$ light curves provide repeated but non-identical views of the source. Periodic or coherent behaviour may appear across bands while stochastic amplitudes and photometric precision differ.

A multiband inference can therefore be more powerful than independent single-band analyses, but only if the model respects different cadences, errors and possible band-dependent variability. This naturally motivates latent-process and modern amortized models later in the course.

## Deep dive — frequency-domain and covariance-domain views

A stationary stochastic process can be described through a covariance function in the time domain or a power spectral density in the frequency domain. These are Fourier-related views of the same second-order structure. A DRW has a characteristic break timescale; DHO-like processes can produce richer spectral shapes.

This duality helps interpret GP kernels physically: a kernel is not merely a smoothness device. It implies a particular spectrum of variability.

### The window function

If $w(t)$ describes the sampling, the observed spectral structure is related to the convolution of the intrinsic spectrum with the spectral window. Seasonal gaps can therefore produce repeated alias structures. Plotting the periodogram of a constant or simple signal sampled at the actual observation times can reveal these features.

### Searching frequencies creates multiplicity

A period search often tests thousands of frequencies. The maximum peak statistic has a different null distribution from the power at one pre-specified frequency. Analytic false-alarm formulas depend on assumptions; simulation from the fitted stochastic null is often safer.

### Model comparison for periodicity

A useful modelling ladder is:

$$
M_0:\text{stochastic only},
\qquad
M_1:\text{stochastic + periodic}.
$$

The periodic component introduces amplitude, phase and period parameters. The stochastic component has its own hyperparameters. If the red-noise amplitude/timescale is fixed incorrectly, the periodic component can absorb stochastic structure and appear significant.

For simulated objects, posterior predictive or repeated-simulation calibration lets students directly measure false-positive rates under $M_0$.

## Binary-SMBH synthesis questions

Answer verbally before leaving the lecture.

1. Which quantity in today's topic is a **property of the simulator**, and which could be an **observable property of a real survey**?
2. What uncertainty or selection effect would most strongly distort the result?
3. What diagnostic would you require before trusting the inference?
4. What information would be unavailable if these were real LSST sources rather than simulations?

## Key equations / ideas to retain

Create your own 6–10 line summary here during class.

## Further reading

- Crete `08_Gaussian_Processes/Gaussian_Processes.ipynb`.
- Rasmussen & Williams, *Gaussian Processes for Machine Learning*.
- VanderPlas on Lomb-Scargle periodograms.

# Part II — Extended lecture development

## 15. Frequency resolution and baseline

A finite observing baseline $T$ limits frequency resolution roughly to
scales of order $1/T$. Closely spaced frequencies can become difficult to
distinguish.

This does not mean "period uncertainty = 1/T" universally. Sampling, S/N,
signal shape and stochastic background all matter. But it provides a useful
sanity scale.

A candidate period comparable to or longer than the baseline is especially
fragile because only a small fraction of a cycle may be observed.

## 16. Weighted sinusoidal fitting

When errors are heteroscedastic, sinusoidal fitting should use weights

$$
w_i=\frac1{\sigma_i^2}.
$$

The frequency scan then compares weighted residual sums or likelihoods.

This can materially change the periodogram when low-quality epochs would
otherwise dominate through sheer number.

A general lesson: periodograms are model fits in disguise, so the measurement
model still matters.

## 17. False-alarm probability under a stochastic null

A practical simulation calibration is:

1. fit a stochastic null model $M_0$;
2. generate many mock light curves using the **actual observation times and
   magnitude errors**;
3. run exactly the same period search on each mock;
4. store the maximum periodogram peak;
5. compare the observed maximum with the empirical null distribution.

Then

$$
\hat p
=
\frac{1+\#(T_b\ge T_{\rm obs})}{B+1}.
$$

This automatically accounts for the scanned frequency grid and cadence under
the chosen null model.

In [ ]:
# Toy maximum-peak null calibration (white noise example only)
rng=np.random.default_rng(2026)
obs_max=power.max()
null_max=[]
for _ in range(300):
    yn=rng.normal(0,err)
    pp=lombscargle(t,yn-yn.mean(),2*np.pi*freq,normalize=True)
    null_max.append(pp.max())

p=(1+np.sum(np.array(null_max)>=obs_max))/(len(null_max)+1)
print("white-noise max-peak p:",p,
      "(not appropriate for a red-noise AGN null)")

## 18. DRW power spectrum intuition

For an Ornstein–Uhlenbeck/DRW process, the power spectral density has a
Lorentzian-like form with a break near the damping timescale.

At frequencies well below the break, the spectrum flattens; at higher
frequencies it behaves approximately as a steep red-noise spectrum.

This explains why DRW realisations show long smooth excursions and why a finite
segment can mimic part of a sinusoid.

The time-domain covariance and frequency-domain PSD are two representations of
the same second-order stochastic structure.

## 19. GP hyperparameter degeneracy

Amplitude and timescale can be correlated. If the observing baseline is much
shorter than $\tau$, many large-$\tau$ models look similar over the observed
window.

This is an identifiability issue. The GP optimiser or posterior may prefer a
broad ridge rather than a sharply measured timescale.

Students should compare the inferred $\tau$ with:
- baseline;
- cadence;
- simulator truth after fitting;
- posterior/likelihood width.

A timescale far outside the temporal support should not be reported with false
precision.

## 20. Quasi-periodic covariance

A common quasi-periodic kernel combines a periodic factor with finite coherence,
for example schematically

$$
k(\Delta t)
=
A^2
\exp\left[
-\frac{\Delta t^2}{2\lambda^2}
-
\Gamma\sin^2\left(\frac{\pi\Delta t}{P}\right)
\right].
$$

Here $P$ sets a characteristic period while $\lambda$ controls how long
phase coherence persists.

Such a model is fundamentally different from a perfectly coherent sinusoid.
Comparing the two can reveal whether the data require a stable orbital clock or
only recurrent stochastic structure.

## 21. Mean functions and covariance functions can compete

A GP model can include a periodic mean

$$
m(t)=C+A\sin(2\pi t/P+\phi)
$$

plus a stochastic covariance.

If the covariance is too flexible, it can absorb the periodic signal. If it is
too rigid, the mean can absorb stochastic excursions.

Inference should therefore examine posterior dependence between periodic and
stochastic parameters and validate recovery on simulations.

## 22. Time-domain cross-validation

Randomly withholding individual points from a correlated time series is not the
same as forecasting. Neighbouring training points make random interpolation
easy.

More informative validation may hold out:
- contiguous temporal blocks;
- future segments;
- one band;
- one observing season.

The validation split should match the intended claim: interpolation,
forecasting, cross-band prediction or parameter recovery.

## 23. Multiband latent-process view

Rather than fitting each band independently, imagine a shared latent process
$f(t)$ and band-specific transformations:

$$
m_b(t)=\mu_b+a_b f(t-\tau_b)+\epsilon_b(t).
$$

This structure underlies reverberation/lag models and multiband variability
inference. Even if the course does not fully fit such a model, it shows how
Gaussian processes can encode scientifically motivated sharing across bands.

The binary-SMBH simulator therefore supports much richer future projects than
single-band periodograms.

## 24. Week-7 reporting standard

A defensible time-series result should show:
- light curve with uncertainties;
- baseline/cadence summary;
- searched period range and why;
- window/alias diagnostic;
- stochastic null specification;
- period candidate uncertainty;
- global rather than single-frequency significance;
- predictive/residual checks;
- truth recovery only after inference.

This is the standard students should carry into real AGN variability work.

# Part III — Time-domain board problems

## Problem A — convert the documented period units correctly

The provider stores observed orbital period in **years**, but light-curve times
are MJD days.

For stored period $P=2.0$ yr, the approximate comparison period used in the
provider notebook is

$$
P_{\rm days}\approx730\ {\rm d}.
$$

Students should perform this conversion before overlaying a period on a time
axis. Unit mistakes can look like statistical failures.

## Problem B — only 1.5 cycles observed

Suppose the best period is 400 days and the baseline is 600 days. Only 1.5
cycles are covered.

Even if the periodogram has a strong peak, ask:
- can stochastic long-term variability mimic it?
- is phase coherence testable?
- how broad is the likelihood/posterior in period?
- what happens under an extended baseline simulation?

A candidate period should be interpreted relative to the baseline, not only the
peak power.

## Problem C — white-noise versus DRW false alarms

Generate many DRW light curves with no periodic component and calculate the
largest white-noise-calibrated LS peak. If nominal "1%" peaks occur much more
often than 1%, the white-noise significance is miscalibrated.

This experiment is more persuasive than merely telling students that red noise
is dangerous because they measure the false-positive inflation directly.